### create surfaces of topic models derived from street view imagery ###
**Author:** Andrew Larkin <br>
**Organizaation**: Oregon State University, College of Health

### part 1: import libraries and define global static constants ###

In [ ]:
import pandas as ps
import numpy as np
from copy import deepcopy

In [ ]:
PARENT_FOLDER = "H:/"
ID_LINK = PARENT_FOLDER + "MSA_329_link.csv" # file to join location ids with image ids
CORVALLIS_2020_IMGS = PARENT_FOLDER + "BEACON/Corvallis2020Imgs.csv" # metadata of Corvallis street view imagery
CORVALLIS_TOPICS = PARENT_FOLDER + "CorvallisTopics.csv" # predictions from the topic model
COMPARISON_IDS = PARENT_FOLDER + "BEACON/mturk_cate_nature_one_city.csv" # metadata for each comparison
COMPARISON_DIRECTION = PARENT_FOLDER + "extractedDirection.csv"  # LLM outputs (chosen direction and rationale)
TOPICS_BY_LOCATION = PARENT_FOLDER + "geoTopicScoresv2.csv" # topic scores for each location
TOPIC_EXAMPLES = PARENT_FOLDER + "topBottomTopics.csv" # 10 examples for each topic

### part 2: helper functions ###

In [ ]:
# calculate the number of instances of each topic for each location
def calcNTopics():
    
    # part 1: join files
    idLink = ps.read_csv(ID_LINK) 
    Corvallis2020 = ps.read_csv(CORVALLIS_2020_IMGS)
    joined = ps.merge(idLink,Corvallis2020,how='inner',on='panId')
    comparisons = ps.read_csv(COMPARISON_IDS)
    comparison_direction = ps.read_csv(COMPARISON_DIRECTION)
    topics = ps.read_csv(CORVALLIS_TOPICS)
    topics = topics[topics['score']>=0.7] # screen for topic predictions with a prob >= 0.7 (i.e. high confidence)
    joined2 = ps.merge(comparisons,comparison_direction,how='inner',left_on='index',right_on='comparisonNum')
    
    # get topic predictions for the left image in each comparison
    joined2left = joined2[joined2['direction']=='left']
    joined2leftWin = deepcopy(joined2left) # topics where the left image won
    joined2leftLose = deepcopy(joined2left) # topics where the left image lost
    
    # remove folderpath from filename
    joined2leftWin['img'] = joined2left['l_img'].str[-20:]
    joined2leftLose['img'] = joined2left['r_img'].str[-20:]
     
    # get topic predictions for the right image in each comparison
    joined2right = joined2[joined2['direction']=='right']
    joined2rightWin = deepcopy(joined2right)
    joined2rightLose = deepcopy(joined2right)
    
    # remove folderpath from filename
    joined2rightWin['img'] = joined2right['r_img'].str[-20:]
    joined2rightLose['img'] = joined2right['l_img'].str[-20:]
    
    # join topics of winning images (left and right)
    joined2Win = ps.concat([joined2leftWin,joined2rightWin])
    joined2Win = ps.merge(joined2Win,topics,how='inner',left_on='comparisonNum',right_on='comparisonNum')
    
    # join topics of losing images (left and right)
    joined2Lose = ps.concat([joined2leftLose,joined2rightLose])
    joined2Lose = ps.merge(joined2Lose,topics,how='inner',left_on='comparisonNum',right_on='comparisonNum')
    
    # multiply losing image score by -1 (i.e. losing images reduce the topic score for the losing location)
    joined2Lose['score'] = joined2Lose['score']*-1

    
    joined2 = ps.concat([joined2Win,joined2Lose])
    joined3 = ps.merge(joined2,joined,how='inner',left_on='img',right_on='filename')
    joined4 = joined3[['img','panId','comparisonNum']]
    joined4.drop_duplicates(inplace=True)
    joined3 = joined3[['panId','topic','score']]
    stats = joined3.groupby(['panId','topic'],as_index=False).sum()
    stats2 = joined4.groupby(['panId'],as_index=False).count()
    stats2 = stats2[['panId','comparisonNum']]
    stats = ps.merge(stats,stats2,how='inner',on='panId')
    statsArr = []
    fatColumn = ps.DataFrame({
        'panId':list(set(stats['panId']))
    })
    
    # for each topic vector in the topic model and each location id, sum the number of win and lose instances
    # and normalize by the number of comparisons
    for index in range(25):
        tempData = stats[stats['topic']==index]
        tempData['topic' + str(index)] = tempData['score']/tempData['comparisonNum']
        tempData = tempData[['panId','topic' + str(index)]]
        fatColumn = ps.merge(fatColumn,tempData,how='left',on='panId')
        
    # join comparisons with location metadata and save as csv
    joined = joined[['panId','imgLat','imgLon']]
    joined.drop_duplicates(inplace=True)
    fatColumn.fillna(0, inplace=True)
    fatColumn = ps.merge(fatColumn,joined,how='inner',on='panId')
    fatColumn.to_csv(TOPICS_BY_LOCATION,index=False)

In [ ]:
# for a given topic, screen for exampples of the plain text LLM rationale used to classify topics
# INPUTS:
#    topicNum (int) - topic to get 10 examples for
#    topics (pandas df) - topic classifications for each image comparison
#    text (pandas df) - plain text rationale for each image comparison
# OUTPUTS:
#    dfArr (pandas df) - 10 examples for the desired topic
def findTopicExamples(topicNum,topics,text):
    
    # get the index numbers of 10 comparisons from the topics dataset for those classified as the desired dataset
    screenedTopics = topics[topics['topic']==topicNum] 
    compareNums = list(set((screenedTopics[screenedTopics['score']>0.8])['comparisonNum']))[0:10]
    dfArr = []
    
    # join data from the topics and text datasets and return as a pandas dataframe
    for compare in compareNums:
        compareTopics = topics[topics['comparisonNum']==compare]
        compareTopics['lines'] = list(range(compareTopics.count().iloc[0]))
        compareTopics = compareTopics[compareTopics['topic']==topicNum]
        compareText = text[text['comparisonNum']==compare]
        compareText['lines'] = list(range(compareText.count().iloc[0]))
        selectedText = ps.merge(compareTopics,compareText,how='inner',on='lines')
        selectedText = selectedText[['comparisonText','topic','score','comparisonNum_x']]
        dfArr.append(selectedText)
    return(ps.concat(dfArr))


### main function ###

In [ ]:
# calculate topic scores for each locaiton
calcNTopics()

# read topic scores and comparison metadata into memory
topics = ps.read_csv(CORVALLIS_TOPICS)
comparisons = ps.read_csv(PARENT_FOLDER + "CorvallisComparisons.csv")
exampleArr = []

# for each topic vector, get examples and save to csv
for topicNum in range(25):
    exampleArr.append(findTopicExamples(topicNum,topics,comparisons))
exampleDF = ps.concat(exampleArr)
exampleDF.to_csv(TOPIC_EXAMPLES,index=False)